In [8]:
%load_ext autoreload
%autoreload 2

import os
import random
import numpy as np
import pandas as pd
import sys
import torch
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def find_root(current_path, marker="setup.py"):
    current_path = Path(current_path).resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / marker).exists():
            return parent
    return current_path

PROJECT_ROOT = find_root(Path.cwd())

DATASETS_DIR = PROJECT_ROOT / "data" / "datasets_clean" / "generalization"
RESULTS_DIR = PROJECT_ROOT / "data" / "results_clean" / "generalization"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Add project root to the Python path
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from rl_methods.mdp_clean import DiscreteMDP, Planner
from rl_methods.fogas_clean import FOGASEvaluator, FOGASHyperOptimizer
from rl_methods.fogas_generalization_clean import *
from rl_methods.fogas_generalization_clean import (
    FinalLinearSolver,
    TabularFeatures,
    LinearFunction,
    LinearQFunction,
)
from rl_methods.data_collection_clean import DiscreteDataBuffer
from rl_methods.fogas_clean import FOGASEvaluator

from rl_methods.fogas_clean import (
    FOGASSolver,
    FOGASOracleSolver,
    FOGASHyperOptimizer,
    FOGASEvaluator,
    FOGASDataset,
)

from rl_methods.fqi_clean import FQISolver

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Problem

In [20]:
states = torch.arange(100, dtype=torch.long)
actions = torch.arange(4, dtype=torch.long)

N = len(states)
A = len(actions)
gamma = 0.9
x_0 = 0

dataset_path_10grid = str(DATASETS_DIR / "10grid_tabular.csv")

grid_size = 10
goal_grid = 99

pit_grids = {18, 32, 57, 61, 75}

wall_states = {
    4, 11, 14, 17, 21, 22, 27, 34, 37,
    40, 42, 43, 44, 45, 46, 47, 49,
    54, 62, 64, 66, 72, 76, 82, 84, 86, 87, 94
}

terminal_states = {goal_grid, *pit_grids}


def state_to_pos(s):
    return divmod(int(s), grid_size)


def pos_to_state(row, col):
    return row * grid_size + col


def move_deterministic(s, a):
    s = int(s)
    a = int(a)

    if s in terminal_states:
        return s

    row, col = state_to_pos(s)

    if a == 0:      # Up
        new_row, new_col = row - 1, col
    elif a == 1:    # Down
        new_row, new_col = row + 1, col
    elif a == 2:    # Left
        new_row, new_col = row, col - 1
    elif a == 3:    # Right
        new_row, new_col = row, col + 1
    else:
        raise ValueError("action must be in {0, 1, 2, 3}")

    if not (0 <= new_row < grid_size and 0 <= new_col < grid_size):
        return s

    sp = pos_to_state(new_row, new_col)

    if sp in wall_states:
        return s

    return sp


def transition_probs(s, a, intended_prob=1.0):
    probs_by_state = {}

    for candidate_a in range(A):
        prob = (1.0 - intended_prob) / A

        if candidate_a == int(a):
            prob += intended_prob

        sp = move_deterministic(s, candidate_a)
        probs_by_state[sp] = probs_by_state.get(sp, 0.0) + prob

    return probs_by_state


def transition_fn(s, a):
    probs = torch.zeros(N, dtype=torch.float64)

    for sp, prob in transition_probs(s, a).items():
        probs[sp] = prob

    return probs


def next_state(s, a):
    """
    Sample one next state from the stochastic transition.
    Useful for manual rollouts, not used by DiscreteMDP planning.
    """
    probs = transition_fn(s, a)
    return int(torch.multinomial(probs.float(), num_samples=1).item())


def reward_from_next_state(sp):
    sp = int(sp)

    if sp == goal_grid:
        return 1.0

    if sp in pit_grids:
        return -5.0

    return -0.1


def reward_fn(s, a):
    """
    Expected one-step reward under P(. | s, a).
    """
    if int(s) in terminal_states:
        return 0.0

    return sum(
        prob * reward_from_next_state(sp)
        for sp, prob in transition_probs(s, a).items()
    )


mdp = DiscreteMDP(
    states=states,
    actions=actions,
    gamma=gamma,
    x0=x_0,
    reward_fn=reward_fn,
    transition_fn=transition_fn,
    terminal_states=list(terminal_states),
).to(DEVICE)

planner = Planner(mdp)

## Data collection

In [ ]:
# Initialize the clean collector
collector = DiscreteDataBuffer(
    mdp=mdp,
    reset_probs={"custom": 1.0},
    initial_states=[x_0],       # required for custom resets
    max_steps=50,
    terminal_states={goal, *pits},
    seed=seed,
)

# Epsilon-greedy around optimal policy.
# NOTE: epsilon=0.3 means 70% optimal, 30% random.
epsilon_policy = (planner.pi_star, 0.3)

# Mixed collection, equivalent to old collect_mixed_dataset
df = collector.collect(
    policies=[epsilon_policy, "random"],
    proportions=[0.8, 0.2],
    n_steps=8000,
    extra_terminal_steps=3,
    episode_based=True,
    save_path=str(DATASET_PATH),
    verbose=True,
)

# FOGAS

In [21]:
from rl_methods.fogas_clean import FOGASSolver, FOGASEvaluator, FOGASDataset

# Tabular state-action features: one-hot over (s, a)
def phi(s, a):
    feat = torch.zeros(N * A, dtype=torch.float64)
    feat[int(s) * A + int(a)] = 1.0
    return feat


# Since phi is one-hot, omega[idx] is exactly r(s, a).
# Here r(s, a) is your expected one-step reward under the stochastic transition.
omega = torch.empty(N * A, dtype=torch.float64)
for s in range(N):
    for a in range(A):
        omega[s * A + a] = reward_fn(s, a)


# Rebuild the same MDP, but expose phi/omega so FOGAS uses known rewards
# instead of estimating omega from the dataset.
mdp_fogas = DiscreteMDP(
    states=states,
    actions=actions,
    gamma=gamma,
    x0=x_0,
    omega=omega,
    phi=phi,
    transition_fn=transition_fn,
    terminal_states=list(terminal_states),
).to(DEVICE)

planner_fogas = Planner(mdp_fogas)

# Same style as experiments/fogas_clean/notebooks/10grid_tabular.ipynb
FOGAS_T = 20_000
FOGAS_BETA = 1e-7

# Good values from the clean 10-grid tabular search range.
# If you want exact reproduction, run the grid script mentioned below.
FOGAS_ALPHA = 0.005
FOGAS_ETA = 0.002
FOGAS_RHO = 0.0005
FOGAS_D_THETA = 60.0

solver = FOGASSolver(
    mdp=mdp_fogas,
    phi=phi,
    csv_path=dataset_path_10grid,
    device=DEVICE,
    seed=42,
    beta=FOGAS_BETA,
    print_params=True,
)

pi_fogas = solver.run(
    T=FOGAS_T,
    alpha=FOGAS_ALPHA,
    eta=FOGAS_ETA,
    rho=FOGAS_RHO,
    D_theta=FOGAS_D_THETA,
    tqdm_print=True,
)

evaluator = FOGASEvaluator(
    solver=solver,
    mdp=mdp_fogas,
    planner=planner_fogas,
)

greedy_pi_fogas = evaluator.greedy_policy(pi_fogas)

v_fogas, q_fogas = planner_fogas.evaluate_policy(pi_fogas)
v_greedy, q_greedy = planner_fogas.evaluate_policy(greedy_pi_fogas)

print("Optimal normalized return:", planner_fogas.optimal_policy_return())
print("FOGAS normalized return:  ", planner_fogas.policy_return(pi_fogas))
print("Greedy normalized return: ", planner_fogas.policy_return(greedy_pi_fogas))

print(
    "FOGAS success rate:",
    evaluator.success_rate(
        goal_state=goal_grid,
        policy_mode="solver",
        num_trajectories=100,
        max_steps=50,
        seed=42,
        terminal_states=terminal_states,
    )["policy"],
)

print(
    "Greedy success rate:",
    evaluator.success_rate(
        goal_state=goal_grid,
        policy_mode="greedy",
        num_trajectories=100,
        max_steps=50,
        seed=42,
        terminal_states=terminal_states,
    )["policy"],
)


Device: cuda
Dataset: /shared/home/mauro.diaz/work/FOGAS/data/datasets_clean/generalization/10grid_tabular.csv (n=8000)

================ FOGAS PARAMETER SUMMARY ================

Basic Information
-----------------
Dataset size n:           8000
Feature norm bound R:     1.0000
Num states N:             100
Num actions A:            4
Feature dim d:            400
Discount γ:               0.9
Confidence δ:             0.05

Theoretical Quantities
----------------------
T_min (theoretical):      7404.102821112293
T (iterations):                7405

FOGAS Hyperparameters
---------------------
alpha:                        0.000097
rho:                            2555.623765
eta:                            0.000001
D_theta:                    63.245553
beta (ridge):             0.000000   (overridden → 0.000000)
D_pi (derived):           45.311168




FOGAS: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20000/20000 [00:29<00:00, 685.42it/s]


Optimal normalized return: -0.08227412376069498
FOGAS normalized return:   -0.08730763236130935
Greedy normalized return:  -0.08227412376069498
FOGAS success rate: 0.79
Greedy success rate: 1.0


# FQI

In [22]:
from rl_methods.fqi_clean import FQISolver
from rl_methods.fogas_clean import FOGASEvaluator

SEED = 42

solver_fqi = FQISolver(
    mdp=mdp_fogas,
    phi=phi,
    csv_path=str(dataset_path_10grid),
    planner=planner_fogas,
    device=DEVICE,
    seed=SEED,
    ridge=1e-2,
    augment_terminal_transitions=True,
)

_ = solver_fqi.run(
    K=1000,
    tau=0.1,
    verbose=True,
)

pi_fqi = solver_fqi.pi
greedy_pi_fqi = evaluator_fqi.greedy_policy(pi_fqi) if False else None

evaluator_fqi = FOGASEvaluator(
    solver=solver_fqi,
    mdp=mdp_fogas,
    planner=planner_fogas,
)

greedy_pi_fqi = evaluator_fqi.greedy_policy(pi_fqi)

v_fqi, q_fqi = planner_fogas.evaluate_policy(pi_fqi)
v_greedy_fqi, q_greedy_fqi = planner_fogas.evaluate_policy(greedy_pi_fqi)

print("Optimal normalized return:", planner_fogas.optimal_policy_return())
print("FQI normalized return:    ", planner_fogas.policy_return(pi_fqi))
print("Greedy normalized return: ", planner_fogas.policy_return(greedy_pi_fqi))

print(
    "FQI success rate:",
    evaluator_fqi.success_rate(
        goal_state=goal_grid,
        policy_mode="solver",
        num_trajectories=100,
        max_steps=50,
        seed=SEED,
        terminal_states=terminal_states,
    )["policy"],
)

print(
    "Greedy success rate:",
    evaluator_fqi.success_rate(
        goal_state=goal_grid,
        policy_mode="greedy",
        num_trajectories=100,
        max_steps=50,
        seed=SEED,
        terminal_states=terminal_states,
    )["policy"],
)

FQI: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:00<00:00, 1704.46it/s, theta_norm=233.6305]


Optimal normalized return: -0.08227412376069498
FQI normalized return:     -0.08227412376069498
Greedy normalized return:  -0.08227412376069498
FQI success rate: 1.0
Greedy success rate: 1.0


# SBEED

In [23]:
import pandas as pd
import torch

from rl_methods.sbeed.datasets import DiscreteSBEEDDataset
from rl_methods.sbeed.features.discrete_features import (
    TabularStateFeatures,
    TabularStateActionFeatures,
    LinearValueParam,
    LinearRhoParam,
    SoftmaxLinearPolicyParam,
)
from rl_methods.sbeed.solvers import DiscreteSBEED
from rl_methods.fogas_clean import FOGASEvaluator


# Use the same MDP/planner objects as FOGAS/FQI evaluation.
# If your notebook only has mdp/planner, replace mdp_fogas/planner_fogas below.
eval_mdp = mdp_fogas
eval_planner = planner_fogas

SEED = 42

df = pd.read_csv(dataset_path_10grid)

done = df["next_state"].astype(int).isin(terminal_states).to_numpy()

dataset = DiscreteSBEEDDataset(
    X=df["state"].to_numpy(),
    A=df["action"].to_numpy(),
    R=df["reward"].to_numpy(),
    X_next=df["next_state"].to_numpy(),
    D=done,
)

value_features = TabularStateFeatures(N)
rho_features = TabularStateActionFeatures(N, A)

solver_sbeed = DiscreteSBEED(
    n_states=N,
    n_actions=A,
    gamma=gamma,
    value_param=LinearValueParam(value_features, N),
    rho_param=LinearRhoParam(rho_features, N, A),
    policy_param=SoftmaxLinearPolicyParam(value_features, N, A),

    lambda_entropy=0.0001,
    eta=1.0,

    lr_value=3e-3,
    lr_rho=1e-3,
    lr_policy=3e-3,

    tau=500.0,
    max_buffer_size=len(df),
    batch_size=None,
    rollout_length=1,

    seed=SEED,
    device=DEVICE,
)

dataset.validate(N, A)
solver_sbeed.dataset = dataset.to(DEVICE)
solver_sbeed.n = solver_sbeed.dataset.n

history = []

print("step    objective      primal_mse      dual_mse")
print("-" * 52)

for t in range(3000):
    stats = solver_sbeed.step()
    history.append(stats)

    if t % 100 == 0 or t == 2999:
        print(
            f"{t:4d}  "
            f"{stats['objective']:12.6f}  "
            f"{stats['primal_mse']:12.6f}  "
            f"{stats['dual_mse']:12.6f}"
        )


# ---------------------------------------------------------------------
# Same evaluation style as the FOGAS/FQI blocks
# ---------------------------------------------------------------------

pi_sbeed = solver_sbeed.get_policy_matrix().to(dtype=torch.float64, device=DEVICE)
solver_sbeed.pi = pi_sbeed

# Attach MDP for compatibility with FOGASEvaluator.
solver_sbeed.mdp = eval_mdp

evaluator_sbeed = FOGASEvaluator(
    solver=solver_sbeed,
    mdp=eval_mdp,
    planner=eval_planner,
)

greedy_pi_sbeed = evaluator_sbeed.greedy_policy(pi_sbeed)

v_sbeed, q_sbeed = eval_planner.evaluate_policy(pi_sbeed)
v_greedy_sbeed, q_greedy_sbeed = eval_planner.evaluate_policy(greedy_pi_sbeed)

print("Optimal normalized return:", eval_planner.optimal_policy_return())
print("SBEED normalized return:  ", eval_planner.policy_return(pi_sbeed))
print("Greedy normalized return: ", eval_planner.policy_return(greedy_pi_sbeed))

print(
    "SBEED success rate:",
    evaluator_sbeed.success_rate(
        goal_state=goal_grid,
        policy_mode="solver",
        num_trajectories=100,
        max_steps=50,
        seed=SEED,
        terminal_states=terminal_states,
    )["policy"],
)

print(
    "Greedy success rate:",
    evaluator_sbeed.success_rate(
        goal_state=goal_grid,
        policy_mode="greedy",
        num_trajectories=100,
        max_steps=50,
        seed=SEED,
        terminal_states=terminal_states,
    )["policy"],
)

step    objective      primal_mse      dual_mse
----------------------------------------------------
   0     -0.001165      0.807389      0.808554
 100     -0.129761      0.702284      0.832045
 200     -0.288864      0.630248      0.919112
 300     -0.470138      0.580275      1.050413
 400     -0.663223      0.545082      1.208305
 500     -0.861049      0.520034      1.381083
 600     -1.059236      0.502137      1.561373
 700     -1.255147      0.489414      1.744561
 800     -1.447238      0.480529      1.927767
 900     -1.634635      0.474547      2.109182
1000     -1.816886      0.470802      2.287688
1100     -1.993799      0.468806      2.462605
1200     -2.165351      0.468195      2.633546
1300     -2.331622      0.468692      2.800315
1400     -2.492756      0.470085      2.962840
1500     -2.648932      0.472206      3.121137
1600     -2.800351      0.474922      3.275273
1700     -2.947224      0.478128      3.425353
1800     -3.089762      0.481739      3.571501
1900  

In [24]:
evaluator_sbeed.print_solver_policy()

  State 0: pi(a=0|s=0) = 0.18  pi(a=1|s=0) = 0.15  pi(a=2|s=0) = 0.14  pi(a=3|s=0) = 0.52  --> best action: 3
  State 1: pi(a=0|s=1) = 0.14  pi(a=1|s=1) = 0.17  pi(a=2|s=1) = 0.16  pi(a=3|s=1) = 0.53  --> best action: 3
  State 2: pi(a=0|s=2) = 0.11  pi(a=1|s=2) = 0.62  pi(a=2|s=2) = 0.12  pi(a=3|s=2) = 0.15  --> best action: 1
  State 3: pi(a=0|s=3) = 0.22  pi(a=1|s=3) = 0.37  pi(a=2|s=3) = 0.22  pi(a=3|s=3) = 0.20  --> best action: 1
  State 4: pi(a=0|s=4) = 0.25  pi(a=1|s=4) = 0.25  pi(a=2|s=4) = 0.25  pi(a=3|s=4) = 0.25  --> best action: 0
  State 5: pi(a=0|s=5) = 0.10  pi(a=1|s=5) = 0.12  pi(a=2|s=5) = 0.12  pi(a=3|s=5) = 0.67  --> best action: 3
  State 6: pi(a=0|s=6) = 0.07  pi(a=1|s=6) = 0.07  pi(a=2|s=6) = 0.07  pi(a=3|s=6) = 0.79  --> best action: 3
  State 7: pi(a=0|s=7) = 0.04  pi(a=1|s=7) = 0.08  pi(a=2|s=7) = 0.08  pi(a=3|s=7) = 0.79  --> best action: 3
  State 8: pi(a=0|s=8) = 0.07  pi(a=1|s=8) = 0.14  pi(a=2|s=8) = 0.07  pi(a=3|s=8) = 0.71  --> best action: 3
  State 9:

# Generalization

In [25]:
import torch

from rl_methods.fogas_clean import FOGASEvaluator
from rl_methods.fogas_generalization_clean import (
    FinalLinearSolver,
    LinearFunction,
    LinearQFunction,
    TabularFeatures,
)

SEED = 42

# Use the same evaluation MDP/planner as the FOGAS, FQI, and SBEED blocks.
# If you did not define mdp_fogas/planner_fogas, replace these with mdp/planner.
eval_mdp = mdp_fogas
eval_planner = planner_fogas

# ---------------------------------------------------------------------
# Best params from:
# data/results_clean/generalization/final_linear_10grid_tabular_grid_search_best.csv
# ---------------------------------------------------------------------

GEN_ALPHA = 0.003
GEN_ETA = 0.001
GEN_RHO = 2.0
GEN_T = 4000

GEN_THETA_LR = 0.01
GEN_THETA_INNER_STEPS = 20
GEN_THETA_LAMBDA = 1e-6

GEN_REINFORCE_SAMPLES = 4


# ---------------------------------------------------------------------
# Generalized FOGAS tabular features
# ---------------------------------------------------------------------

u_features = TabularFeatures(N, A)
q_features = TabularFeatures(N, A)
policy_features = TabularFeatures(N, A)


solver_gen_fogas = FinalLinearSolver(
    n_states=N,
    n_actions=A,
    gamma=gamma,
    x0=x_0,
    csv_path=str(dataset_path_10grid),

    u_function=LinearFunction(u_features),
    q_function=LinearQFunction(q_features),
    policy_features=policy_features,

    seed=SEED,
    device=DEVICE,

    theta_mode="reg_fixed",
    theta_optimizer="adam",
    theta_start_mode="warm",
    theta_include_beta_cov=False,
    theta_lr=GEN_THETA_LR,
    theta_inner_steps=GEN_THETA_INNER_STEPS,
    theta_lambda=GEN_THETA_LAMBDA,

    beta_update="fogas_full",
)

pi_gen_fogas = solver_gen_fogas.run(
    alpha=GEN_ALPHA,
    eta=GEN_ETA,
    rho=GEN_RHO,
    T=GEN_T,

    theta_lr=GEN_THETA_LR,
    theta_inner_steps=GEN_THETA_INNER_STEPS,
    theta_lambda=GEN_THETA_LAMBDA,

    policy_optimizer="adam",
    policy_gradient="exact",
    reinforce_samples=GEN_REINFORCE_SAMPLES,
    state_weight_update="normal",

    tqdm_print=True,
    verbose=False,
)

# Attach MDP for evaluator compatibility.
solver_gen_fogas.mdp = eval_mdp

evaluator_gen_fogas = FOGASEvaluator(
    solver=solver_gen_fogas,
    mdp=eval_mdp,
    planner=eval_planner,
)

greedy_pi_gen_fogas = evaluator_gen_fogas.greedy_policy(pi_gen_fogas)

v_gen_fogas, q_gen_fogas = eval_planner.evaluate_policy(pi_gen_fogas)
v_greedy_gen_fogas, q_greedy_gen_fogas = eval_planner.evaluate_policy(greedy_pi_gen_fogas)

print("Optimal normalized return:", eval_planner.optimal_policy_return())
print("Gen FOGAS normalized return:", eval_planner.policy_return(pi_gen_fogas))
print("Greedy normalized return:   ", eval_planner.policy_return(greedy_pi_gen_fogas))

print(
    "Gen FOGAS success rate:",
    evaluator_gen_fogas.success_rate(
        goal_state=goal_grid,
        policy_mode="solver",
        num_trajectories=100,
        max_steps=50,
        seed=SEED,
        terminal_states=terminal_states,
    )["policy"],
)

print(
    "Greedy success rate:",
    evaluator_gen_fogas.success_rate(
        goal_state=goal_grid,
        policy_mode="greedy",
        num_trajectories=100,
        max_steps=50,
        seed=SEED,
        terminal_states=terminal_states,
    )["policy"],
)

diagnostics = solver_gen_fogas.get_diagnostics()
if diagnostics:
    final_diag = diagnostics[-1]
    print("Final total loss:", final_diag["total_loss"])
    print("Final policy objective:", final_diag["policy_objective"])
    print("Final beta objective:", final_diag["beta_objective"])
    print("Final q objective:", final_diag["q_objective"])

FinalLinearSolver: 100%|█████████████████████████████████████████████████████████████████████| 4000/4000 [00:46<00:00, 85.96it/s, beta=3.049e+00, policy=-2.306e+01, q=1.344e-01, total_loss=5.215e-02]


Optimal normalized return: -0.08227412376069498
Gen FOGAS normalized return: -0.0971744577091963
Greedy normalized return:    -0.08227412376069498
Gen FOGAS success rate: 0.79
Greedy success rate: 1.0
Final total loss: 0.052148831924634376
Final policy objective: -23.058003479992983
Final beta objective: 3.049424450030787
Final q objective: 0.1343987722583669
